In [ ]:
import os, sys, json, re, time, subprocess
from pathlib import Path

try:
    import pandas as pd
    import numpy as np
except Exception:
    import sys as _s, subprocess as _sp
    _sp.check_call([_s.executable, '-m', 'pip', 'install', '-q', 'pandas', 'numpy', 'scikit-learn', 'mlflow', 'fastapi', 'uvicorn', 'joblib', 'requests'])
    import pandas as pd
    import numpy as np

p = Path.cwd()
while not (p / 'churnDataset.csv').exists() and p.parent != p:
    p = p.parent
os.chdir(p)
rt = Path.cwd()
dd = rt / 'data'
md = rt / 'models'
ld = rt / 'logs'
mlr = rt / 'mlruns'
for d in [dd, md, ld]:
    d.mkdir(exist_ok=True)

import mlflow
from mlflow.tracking import MlflowClient

mlflow.set_tracking_uri(mlr.resolve().as_uri())
rs = json.loads((md / 'runs.json').read_text(encoding='utf-8'))
rs = sorted(rs, key=lambda r: r['val_f1'], reverse=True)
mn = 'churn_model'
cl = MlflowClient()
vs = []
for r in rs:
    mv = mlflow.register_model(f"runs:/{r['run_id']}/model", mn)
    vs.append({'name': r['name'], 'version': mv.version, 'val_f1': r['val_f1'], 'test_f1': r['test_f1']})
for i, v in enumerate(vs):
    st = 'Production' if i == 0 else 'Staging'
    try:
        cl.transition_model_version_stage(mn, v['version'], st, archive_existing_versions=False)
    except Exception:
        cl.set_registered_model_alias(mn, st.lower(), v['version'])
    v['stage'] = st
(md / 'registry.json').write_text(json.dumps(vs, indent=2), encoding='utf-8')
print('production', vs[0]['name'], vs[0]['version'], round(vs[0]['val_f1'], 4))
if len(vs) > 1:
    print('compare', vs[0]['name'], 'better than', vs[1]['name'], 'by val_f1')
vs
